In [1]:
#!/usr/bin/env python
# coding: utf-8

"""
Example script (for a Jupyter notebook) demonstrating how to process a CSV file
containing 2D landmark coordinates for multiple frames, and then compute:
- Total movement (frame-to-frame) of each landmark
- Segment-wise breakdown of that movement (initial, remaining, ending)
- Duration-normalized version of each movement measurement

CSV Columns:
    frame
    left_wrist_x, left_wrist_y
    left_index_x, left_index_y
    left_pinky_x, left_pinky_y
    left_thumb_x, left_thumb_y
    right_wrist_x, right_wrist_y
    right_index_x, right_index_y
    right_pinky_x, right_pinky_y
    right_thumb_x, right_thumb_y

Naming Convention:
    The CSV file is named "<video_name>_landmark_coordinates.csv"
    We'll parse out "<video_name>" from that filename.

Usage in a Jupyter notebook:
    - Place this script in a cell.
    - Modify the `csv_dir` to point to the folder containing the CSV files.
    - Run `df_final = process_hand_landmark_csv_directory(csv_dir)`
    - The resulting DataFrame will have one row per CSV, aggregating all
      movement calculations. Then optionally save it to a new CSV file.
"""

import os
import numpy as np
import pandas as pd
from datetime import datetime

# Columns we expect for each landmark set
# These can be easily extended or changed if needed.
LANDMARK_NAMES = [
    'left_wrist', 
    'left_index',
    'left_pinky',
    'left_thumb',
    'right_wrist',
    'right_index',
    'right_pinky',
    'right_thumb'
]

def euclidean_distance_2d(p1, p2):
    """
    Calculates the Euclidean distance between two 2D points p1 and p2.
    Each point is a tuple or list of (x, y).
    """
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def process_pose_csv(
    csv_path,
    fps=30.0,
    initial_percentage=25,
    ending_percentage=25
):
    """
    Processes a single CSV file containing 2D hand landmark coordinates for multiple frames.
    Computes the movement of each landmark by taking Euclidean distances between consecutive frames.

    The CSV must have columns:
        frame
        <landmark>_x, <landmark>_y  for each landmark in LANDMARK_NAMES
    E.g.:
        frame, left_wrist_x, left_wrist_y, left_index_x, left_index_y, ...

    Parameters
    ----------
    csv_path : str
        Path to the .csv file containing 2D landmark coordinates for a single video.
    fps : float, optional
        Frames per second of the original video. Used to compute segment durations and normalization.
    initial_percentage : float, optional
        Percentage (0-100) of the video to treat as "initial" segment.
    ending_percentage : float, optional
        Percentage (0-100) of the video to treat as "ending" segment.

    Returns
    -------
    dict
        A dictionary containing total, segment-wise, and normalized movements for each landmark,
        along with basic metadata like the video name and durations.
        This can easily be turned into a DataFrame row.
    """
    # Extract the "video name" from something like "myvideo_landmark_coordinates.csv"
    basename = os.path.basename(csv_path)
    # Remove the "_landmark_coordinates.csv" part (assuming it always ends with that)
    video_name = basename.replace("_landmark_coordinates.csv", "")

    # Read the CSV
    df = pd.read_csv(csv_path)

    # Sort by frame if not guaranteed to be in order
    df = df.sort_values(by="frame", ascending=True).reset_index(drop=True)

    total_frames = len(df)
    if total_frames <= 1:
        # If there's only one frame, movement will be zero
        # Build and return a default dict with no movements
        return {
            "Video_Name": video_name,
            "Total_Duration": 0.0,
            "Initial_Duration": 0.0,
            "Remaining_Duration": 0.0,
            "Ending_Duration": 0.0,
            # Include zeros for each landmark movement
            **{f"{lm}_movement": 0.0 for lm in LANDMARK_NAMES},
            **{f"{lm}_movement_n": 0.0 for lm in LANDMARK_NAMES},
            **{f"initial_{lm}_movement": 0.0 for lm in LANDMARK_NAMES},
            **{f"initial_{lm}_movement_n": 0.0 for lm in LANDMARK_NAMES},
            **{f"remaining_{lm}_movement": 0.0 for lm in LANDMARK_NAMES},
            **{f"remaining_{lm}_movement_n": 0.0 for lm in LANDMARK_NAMES},
            **{f"ending_{lm}_movement": 0.0 for lm in LANDMARK_NAMES},
            **{f"ending_{lm}_movement_n": 0.0 for lm in LANDMARK_NAMES},
        }

    # Compute total video duration in seconds
    duration = total_frames / fps

    # Segment durations
    initial_duration = (initial_percentage / 100.0) * duration
    ending_duration = (ending_percentage / 100.0) * duration
    remaining_duration = duration - (initial_duration + ending_duration)

    # Convert those durations to frame indices
    initial_frames = int(initial_duration * fps)
    ending_frames = int(ending_duration * fps)
    # We don't specifically need the leftover frames count, but we keep it conceptually
    # remaining_frames = total_frames - (initial_frames + ending_frames)

    # Arrays to accumulate total movement
    # We'll have one index per landmark in LANDMARK_NAMES
    num_landmarks = len(LANDMARK_NAMES)
    total_movements = np.zeros(num_landmarks)
    initial_movements = np.zeros(num_landmarks)
    remaining_movements = np.zeros(num_landmarks)
    ending_movements = np.zeros(num_landmarks)

    # We need to track the previous frame's 2D coords
    # We'll do so for each landmark
    prev_coords = None

    # Iterate from the first actual frame (index 0) up to the last
    for i in range(total_frames):
        # Extract the current frame's 2D coordinates for each landmark
        current_coords = []
        for lm in LANDMARK_NAMES:
            x_col = f"{lm}_x"
            y_col = f"{lm}_y"
            x_val = df.loc[i, x_col]
            y_val = df.loc[i, y_col]
            current_coords.append((x_val, y_val))

        # If there's a previous frame, compute Euclidean distances
        if prev_coords is not None:
            # Movement between consecutive frames for each landmark
            frame_movements = [
                euclidean_distance_2d(curr, prev)
                for curr, prev in zip(current_coords, prev_coords)
            ]

            # Determine which segment this frame belongs to (based on index)
            # We'll use 'i' as the "current" frame index
            if i <= initial_frames:
                initial_movements += frame_movements
            elif i > (total_frames - ending_frames):
                ending_movements += frame_movements
            else:
                remaining_movements += frame_movements

            # Accumulate to total
            total_movements += frame_movements

        # Update previous coords
        prev_coords = current_coords

    # Handle potential zero durations
    # (In case initial_percentage or ending_percentage are 0, or if frames/fps are off)
    if duration == 0: 
        duration = 1e-6
    if initial_duration == 0: 
        initial_duration = 1e-6
    if remaining_duration == 0: 
        remaining_duration = 1e-6
    if ending_duration == 0: 
        ending_duration = 1e-6

    # Normalize
    total_movements_n = total_movements / duration
    initial_movements_n = initial_movements / initial_duration
    remaining_movements_n = remaining_movements / remaining_duration
    ending_movements_n = ending_movements / ending_duration

    # Build the output dictionary
    output_entry = {
        "Video_Name": video_name,
        "Total_Duration": duration,
        "Initial_Duration": initial_duration,
        "Remaining_Duration": remaining_duration,
        "Ending_Duration": ending_duration
    }

    # Add each landmark's unnormalized and normalized movements
    for idx, lm in enumerate(LANDMARK_NAMES):
        # total
        output_entry[f"{lm}_movement"] = total_movements[idx]
        output_entry[f"{lm}_movement_n"] = total_movements_n[idx]
        # initial
        output_entry[f"initial_{lm}_movement"] = initial_movements[idx]
        output_entry[f"initial_{lm}_movement_n"] = initial_movements_n[idx]
        # remaining
        output_entry[f"remaining_{lm}_movement"] = remaining_movements[idx]
        output_entry[f"remaining_{lm}_movement_n"] = remaining_movements_n[idx]
        # ending
        output_entry[f"ending_{lm}_movement"] = ending_movements[idx]
        output_entry[f"ending_{lm}_movement_n"] = ending_movements_n[idx]

    return output_entry


def process_pose_in_csv_directory(
    csv_dir,
    fps=30.0,
    initial_percentage=25,
    ending_percentage=25
):
    """
    Iterates over all CSV files in `csv_dir` whose names end with "_landmark_coordinates.csv",
    processes each with `process_hand_landmark_csv`, and collects the results into a DataFrame.

    Parameters
    ----------
    csv_dir : str
        Directory containing CSV files named like "<video_name>_landmark_coordinates.csv".
    fps : float, optional
        Frames per second used to interpret the CSV's frame data for segment calculations.
    initial_percentage : float, optional
        Percentage (0-100) of the video to treat as "initial" segment.
    ending_percentage : float, optional
        Percentage (0-100) of the video to treat as "ending" segment.

    Returns
    -------
    pd.DataFrame
        A DataFrame with one row per CSV/video, containing total and segment-wise 
        movement calculations for each landmark.
    """
    results = []
    for file_name in os.listdir(csv_dir):
        if file_name.endswith("_landmark_coordinates.csv"):
            csv_path = os.path.join(csv_dir, file_name)
            result = process_pose_csv(
                csv_path=csv_path,
                fps=fps,
                initial_percentage=initial_percentage,
                ending_percentage=ending_percentage
            )
            results.append(result)

    return pd.DataFrame(results)

def save_pose_dataframe_to_csv(
    df,
    initial_percentage=25,
    ending_percentage=25,
    suffix="Experts"
):
    """
    Saves the given DataFrame to a CSV file with a timestamped filename, incorporating
    the initial/ending percentages in the filename.

    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame containing movement results.
    initial_percentage : float
        The initial percentage used in the calculations (0-100).
    ending_percentage : float
        The ending percentage used in the calculations (0-100).
    suffix : str
        A suffix for naming the output CSV file.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"Pose_Movement_{initial_percentage}_{ending_percentage}_{suffix}.csv"
    df.to_csv(output_file, index=False)
    print(f"Output saved to {output_file}")


In [ ]:
df_experts = process_pose_in_csv_directory(
    input_dir="CSV_Files/Experts/Landmark_CSVs",
    initial_percentage=25,
    ending_percentage=25
)
save_pose_dataframe_to_csv(df_experts, 25, 25, suffix="Experts")


In [ ]:
df_novices = process_pose_in_csv_directory(
    input_dir="CSV_Files/Novices/Landmark_CSVs",
    initial_percentage=25,
    ending_percentage=25
)
save_pose_dataframe_to_csv(df_novices, 25, 25, suffix="Novices")
